# Лекција 18: Осигуравање AI агената криптографским рачунима

## Практичан свеска

Ова свеска води кроз четири вежбе:

1. **Потпишите свој први рачун** за позив алата агента и проверите га.
2. **Измени рачун** и посматрајте како верификација не успева.
3. **Направите ланац од три рачуна** и потврдите интегритет ланца.
4. **Омотајте позив алата Microsoft Agent Framework-а** тако да свака радња емитује рачун.

Све криптографске примитиве се увозе из добро одржаваних библиотека (`pynacl` за Ed25519, `jcs` за RFC 8785 канонски JSON, `hashlib` из стандардне Питхон библиотеке за SHA-256). Логика рачуна је обична Питхон која се може читати и мењати.

Покрените ћелије по реду. Сваки одељак је кратак и самосталан.


## Подешавање

Инсталирајте две зависности. Обе имају дозвољавајуће лиценце (Apache-2.0 / MIT).


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Помоћни алати

Ова два помоћна алата се баве base64url кодирањем (без попуне) и SHA-256 хеширањем произвољних објеката. Они задржавају остатак бележнице фокусиран на саму логику примке.


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Одељак 1: Потпишите свој први рачун

Замислите да је наш агент за **Contoso Travel** управо потражио летове од Сиднеја до Лос Анђелеса за једног купца. Желимо да ову позивну операцију алата евидентирамо као потписани рачун тако да будући ревизор може да је провери без поверења у нас.

### Корак 1.1: Креирајте кључ за потписивање

У продукцији, агентов кључ за потписивање би се чувао у хардверском безбедносном модулу (HSM), Azure Key Vault-у или сличном заштићеном складишту. За ову лекцију генеришемо нови кључ у меморији.


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): g3SyD_ecOKa1L8RQ79-pDy9em81H-O_jzp9VG4a3EP0


### Корак 1.2: Креирање података рачуна

Подаци садрже све што желимо да рачун потврди: ко је деловао, којим алатом, са којим аргументима, шта је враћено, под којом политиком и када. Хеширамо аргументе и резултат уместо да их укључујемо директно како рачун не би открио поверљив садржај.


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-08-18T04:55:21Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### Корак 1.3: Потпишите и саставите потврду

Три корака:

1. Канонизујте payload користећи JCS тако да две имплементације које произведу исту логичку потврду произведу идентичан низ бајтова.
2. Потпишите канонске бајтове директно помоћу Ed25519 приватног кључа. PureEdDSA унутарњо хешира поруку, тако да додатни претходни хеш мења протокол.

Потпис се затим прилаже оригиналном payload-у да би се добила коначна потврда.


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-08-18T04:55:21Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "Eu3MmrOSdGq2DuemkjNSPKfz5xbr18okNeTU11NJu2usnsnKtC-pjq2PZM1Oap-WXdVgYkL4iW6SeWPSZ2M9BQ",
    "public_key": "g3SyD_ecOKa1L8RQ79-pDy9em81H-O_jzp9VG4a3EP0"
  }
}


### Корак 1.4: Потврди потврду

Потврда је обрнути процес. Уклонимо потпис, поново израчунамо канонске бајтове и проверавамо потпис у односу на јавни кључ у потврди.

Аудитор који врши ову потврду не треба ништа од нас сем саме потврде. Није потребно позивати неку услугу, упитник кључева или икакво поверење.


In [7]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(canonical, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt)}")

Receipt is valid: True
Pre-hashed receipt valid: False


Требало би да видите `Receipt is valid: True` и `Pre-hashed receipt valid: False`. Позитиван случај доказује да директан-JCS пут потписивања функционише; негативна контрола чини правило опсега потписа извршним уместо да остане као проповијед.  


## Одељак 2: Манипулација рачуном

Цела сврха рачуна је да буду отпорни на манипулацију. Хајде да то докажемо.

Изменићемо тачно један карактер на рачуну и посматрати како ће верификација пропасти.


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### Шта се управо догодило?

Када смо променили `policy_id`, канонски бајтови су се променили. Потпис (који је био преко оригиналних канонских бајтова) више не одговара. Верификација исправно враћа `False`.

Не постоји начин да се модификује било које поље у потврди и да она и даље буде важећа, осим ако нападач нема приватни кључ. Док год је приватни кључ у складишту кључева, а јавни кључ је објављен, немогуће је сакрити злоупотребу.

Пробајте сами: модификујте `tool_name` или `agent_id` или `timestamp` у горе наведеном блоку и поново покрените. Свака измена доводи до неважеће потврде.


## Одељак 3: Повежите потврде у низ

Једна потврда штити једну акцију. Већина агената изводи много акција. Да би цео низ био отпоран на измештање, повезујемо сваку потврду са претходном укључивањем хеша претходне потврде у садржај нове потврде.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Ако неко уклони или преуређује потврду, ланац се прекида управо на тој тачки. Верификација било које касније потврде не успева јер њен `previous_receipt_hash` више не одговара стварном хешу претходника.


In [9]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [10]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:d54d1e138bb080dd6bd825d9f015efc74ad6c7f78ee9504d0b68b39d0733b39c
Receipt 2: tool=confirm_booking, prev=sha256:9fbe5dc2ffcffd7d167ef13a744aecd578e474fb536cb7104ddb1a359607f2f6


In [11]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


Сада прекините ланац тако што ћете изменити средњи рачун и поново верификовати. Измењени рачун не прође проверу потписа, И следећи рачун не прође проверу ланчаног линка (јер његов `previous_receipt_hash` више не одговара хешу измењеног средњег рачуна).


In [12]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


Потврда 0 и даље верификује (није измењена и нема претходника на којег би се ослањала). Потврда 1 не пролази проверу потписа јер смо изменили `tool_args_hash`. Потврда 2 не пролази проверу ланчаног линка јер је њен `previous_receipt_hash` израчунат у односу на оригиналну (сада измењену) потврду 1.

Чак и ако нападач поново потпише измењену потврду 1 (што не може без приватног кључа), неусклађеност ланчаног линка у потврди 2 би и даље открила измену. Да би скрио промену, нападач би морао поново потписати сваку потврду од места измене па надаље, што захтева поседовање приватног кључа.


## Одељак 4: Омотајте позив алата агента са потписивањем рачуна

У стварној примени, не желите да сваки аутор агента памти да позове `make_receipt`. Желите да потписивање рачуна буде аутоматско за сваки позив алата.

Ево најједноставнијег обрасца: класа омотача која узима било коју позивљиву функцију алата и враћа верзију која емитује рачун. Ово се прилагођава било ком оквиру агента, укључујући Microsoft Agent Framework (`agent_framework.foundry`).

Ако немате постављен Microsoft Foundry пројекат, локални мок испод и даље приказује образац.


In [13]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [14]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### Интеграција са Microsoft Agent Framework-ом

Рапрезент `ReceiptedTool` горе је независан од framework-а. Да бисте га користили унутар агента изграђеног помоћу Microsoft Agent Framework-а, региструјте омотану функцију као алат. Скица (заменили бисте mock правом регистрацијом алата у Microsoft Foundry):

```python
# Псеудокод који приказује облик интеграције.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="Ви сте агент Contoso Travel ...",
#     tools=[receipted_lookup],   # омотани алат, не сировa функција
# )
# response = agent.run("Пронађи летове из Сиднеја за Лос Анђелес у јуну.")
#
# # Након извршења, сваки позив алата који је агент направио има потписани рачун:
# audit_chain = receipted_lookup.receipts
```

Аgent framework не треба да зна ништа о потврдама. Потписивање потврда је омотано око алата, а није уграђено у framework. Ово је начин да додате порекло постојећем коду агента без преписивања агента.


## Резиме и изазов за вежбање

Ви сте:

- Генерисали Ed25519 пар кључева.
- Конструисали и потписали потврду за позив агента алата.
- Проверили потврду офлајн користећи само јавни кључ.
- Измарали потврду и приметили да провера није успела.
- Конструисали хеш-ланчани низ од три потврде.
- Измарали средину ланца и приметили и неуспех потписа и неправилност ланца.
- Замотали функцију алата аутоматским потписивањем потврде.

**Изазов за вежбање.** Проширите шему потврде пољем `request_id` (UUID за расподељено праћење). Ажурирајте `make_receipt` да га укључи и потврдите да се потврде и даље проверавају од почетка до краја. Потом измените то поље након потписивања и потврдите да провера не успева. Ово вас приморава да унутарњо разумете како сваки бајт канонског кодирања доприноси потпису.

**Важна граница.** Потврде доказују три ствари и само три ствари: ауторство (овим кључем је потписан овај садржај), интегритет (садржај се није мењао од потписивања) и редослед (ова потврда је дошла након оне потврде). Оне НЕ доказују да је агентова радња била исправна, да је политика названа у `policy_id` заиста процењена или да је агент пратио сва правила. Потврде су темељ. Управљање је систем који градите преко њих.

Поново прочитајте README лекције имајући ту границу на уму. Најчешћа грешка тимова у вези са потврдом је претпоставка да "имамо потврде" значи "ми смо управљани." То није тако. Потврде чине понашање агента проверљивим. Оне не чине да оно буде исправно.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен коришћењем услуге за аутоматски превод [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитативним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразума или погрешна тумачења која произилазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
